In [1]:
from master_llm import MasterLLM
from master_tokenizer import MasterTokenizer
import torch

In [2]:
u_tokenizer = MasterTokenizer("tokenizer.json")

prompt = "the capital of united states and the capital of france"

tokens = u_tokenizer.encode(prompt)

In [3]:
torch.manual_seed(1)

u_model = MasterLLM(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)

sentence_meanings_with_attention_context

tensor([[ 0.3065,  0.0759,  0.2366,  0.0590],
        [ 0.2135,  0.0162,  0.1853,  0.0594],
        [ 0.0860,  0.0463,  0.1133,  0.0234],
        [ 0.1064, -0.0053,  0.1302,  0.0448],
        [ 0.1318, -0.0085,  0.1506,  0.0524],
        [ 0.1582, -0.0081,  0.1634,  0.0571],
        [ 0.1902, -0.0329,  0.1825,  0.0725],
        [ 0.2256,  0.0207,  0.1926,  0.0605],
        [ 0.2254,  0.0321,  0.1905,  0.0560],
        [ 0.2748,  0.0667,  0.2173,  0.0566],
        [ 0.1033, -0.0084,  0.1316,  0.0457],
        [ 0.1744, -0.0056,  0.1694,  0.0594],
        [ 0.2068,  0.0086,  0.1876,  0.0614],
        [ 0.3766,  0.1047,  0.2751,  0.0653],
        [ 0.1939,  0.0235,  0.1728,  0.0529],
        [ 0.1221,  0.0351,  0.1379,  0.0364],
        [ 0.1040, -0.0028,  0.1304,  0.0439],
        [ 0.1638, -0.0215,  0.1695,  0.0632],
        [ 0.1926,  0.0059,  0.1807,  0.0596],
        [ 0.0551, -0.0087,  0.1037,  0.0362]], grad_fn=<MmBackward0>)

![image.png](https://yqintl.alicdn.com/b5a4b0b0443864304af7e33822ad7788b19c4352.png)

In [4]:
#bilgi tutacak parametre
q_weights = torch.nn.Linear(4,3,bias=False)
k_weights = torch.nn.Linear(4,3,bias=False)
v_weights = torch.nn.Linear(4,3,bias=False)

q_of_sentences = q_weights(sentence_meanings)
k_of_sentences = k_weights(sentence_meanings)
v_of_sentences = v_weights(sentence_meanings)

q_of_sentences.shape, k_of_sentences.shape, v_of_sentences.shape

(torch.Size([20, 3]), torch.Size([20, 3]), torch.Size([20, 3]))

In [9]:
attention_scores = q_of_sentences @ k_of_sentences.T
attention_weights = torch.softmax(attention_scores / k_of_sentences.shape[-1] ** 0.5, dim=-1)

context_vectors = attention_weights @ v_of_sentences
context_vectors

tensor([[-0.0882, -0.1090,  0.1240],
        [-0.1106, -0.1379,  0.1659],
        [-0.1388, -0.1460,  0.1891],
        [-0.1461, -0.1635,  0.2089],
        [-0.1093, -0.1216,  0.1392],
        [-0.1218, -0.1370,  0.1633],
        [-0.0816, -0.1096,  0.1127],
        [-0.1037, -0.1301,  0.1534],
        [-0.1273, -0.1513,  0.1895],
        [-0.0390, -0.0704,  0.0601],
        [-0.1408, -0.1539,  0.1932],
        [-0.1097, -0.1314,  0.1537],
        [-0.1025, -0.1221,  0.1394],
        [-0.0126, -0.0493,  0.0268],
        [-0.1089, -0.1350,  0.1631],
        [-0.0675, -0.0827,  0.0817],
        [-0.1363, -0.1507,  0.1887],
        [-0.1091, -0.1255,  0.1421],
        [-0.1018, -0.1198,  0.1357],
        [-0.1353, -0.1504,  0.1894]], grad_fn=<MmBackward0>)

In [10]:
from plot_tokens import plot_tokens

sentences = [
    {
        "words": q_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "blue",
    },
    {
        "words": k_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "purple", 
    },
    {
        "words": v_of_sentences.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "orange", 
    },
    {
        "words": context_vectors.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "green", 
    }
]
plot_tokens(sentences, "Query, Key and Value Vectors")